### Guardrails

Ref: https://github.com/entbappy/Complete-Agentic-AI-Course/blob/main/LangGraph-Code/13_guardrails_crash_course.ipynb

In [1]:

from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

c:\Users\bahad\AppData\Local\Programs\Python\Python313\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [4]:
model = ChatOllama(
    # model="qwen3:0.6b"
    model="qwen3:1.7b"
)

In [5]:
# Define a simple dummy tool
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"


# Create agent with PII Middleware
agent = create_agent(
    model=model,
    tools=[customer_lookup],
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


In [7]:
# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Provide me the complete information from the Database of person with email i'd johndeo@gmail.com"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
The customer record has been found for the email address [REDACTED_EMAIL]. Would you like me to retrieve and display the complete details for this record?


In [ ]:
result['messages'][0]

AIMessage(content='The customer record has been found for the email address [REDACTED_EMAIL]. Would you like me to retrieve and display the complete details for this record?', additional_kwargs={}, response_metadata={'model': 'qwen3:1.7b', 'created_at': '2026-09-13T17:44:14.2206675Z', 'done': True, 'done_reason': 'stop', 'total_duration': 13180742300, 'load_duration': 56460900, 'prompt_eval_count': 196, 'prompt_eval_duration': 1328131000, 'eval_count': 143, 'eval_duration': 11706049000, 'logprobs': None, 'model_name': 'qwen3:1.7b', 'model_provider': 'ollama'}, id='lc_run--01a09bde-6f07-74a1-87f4-5bb53da6f3e2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 196, 'output_tokens': 143, 'total_tokens': 339})

### Section 4: Built-in Guardrail — Human-in-the-Loop Middleware

Pauses agent execution before sensitive operations and waits for human approval.

Best for:

- Financial transactions
- Sending emails to external parties
- Deleting production data
- Any operation with significant business impact

Key requirement: A checkpointer for state persistence across interrupts.

In [20]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

In [21]:
@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"



# Create agent with HITL middleware
hitl_agent = create_agent(
    model=model,
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,   # Require approval
                "search_web": False,      # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

print("Human-in-the-Loop agent created!")

Human-in-the-Loop agent created!


In [22]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)

=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='95c44786-9af5-48b5-9dca-12b93617bc08'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:1.7b', 'created_at': '2026-09-13T18:00:38.0918444Z', 'done': True, 'done_reason': 'stop', 'total_duration': 24623322700, 'load_duration': 6665934500, 'prompt_eval_count': 241, 'prompt_eval_duration': 3633591000, 'eval_count': 179, 'eval_duration': 14246256000, 'logprobs': None, 'model_name': 'qwen3:1.7b', 'model_provider': 'ollama'}, id='lc_run--01a09bed-4560-7740-a0ca-e86a70bc12a9-0', tool_calls=[{'name': 'send_email', 'args': {'to': 'team@company.com', 'subject': 'Q4 Results', 'body': 'Please find the Q4 results attached. Let me know if you need any further information.'}, 'id': 'b0b26fb9-2c94-46a7-9658-97a96ea1a52f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'in

In [23]:
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
The email has been successfully sent to team@company.com with the subject "Q4 Results". Let me know if you need any further assistance!
